In [1]:
# Data handling
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
# Preprocessing tools from scikit-learn
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

In [2]:

df = pd.read_csv(r'C:\Users\SAKTHI\Desktop\myproject\EMI Predict AI\data\emi_prediction_dataset_eda.csv')

In [3]:
# Separate numerical and categorical columns
numerical_cols = df.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_cols = df.select_dtypes(include=['object']).columns.tolist()

print(f"Numerical columns ({len(numerical_cols)}): {numerical_cols}")
print(f"Categorical columns ({len(categorical_cols)}): {categorical_cols}")

Numerical columns (18): ['age', 'monthly_salary', 'years_of_employment', 'monthly_rent', 'family_size', 'dependents', 'school_fees', 'college_fees', 'travel_expenses', 'groceries_utilities', 'other_monthly_expenses', 'current_emi_amount', 'credit_score', 'bank_balance', 'emergency_fund', 'requested_amount', 'requested_tenure', 'max_monthly_emi']
Categorical columns (9): ['gender', 'marital_status', 'education', 'employment_type', 'company_type', 'house_type', 'existing_loans', 'emi_scenario', 'emi_eligibility']


In [4]:
df.columns

Index(['age', 'gender', 'marital_status', 'education', 'monthly_salary',
       'employment_type', 'years_of_employment', 'company_type', 'house_type',
       'monthly_rent', 'family_size', 'dependents', 'school_fees',
       'college_fees', 'travel_expenses', 'groceries_utilities',
       'other_monthly_expenses', 'existing_loans', 'current_emi_amount',
       'credit_score', 'bank_balance', 'emergency_fund', 'emi_scenario',
       'requested_amount', 'requested_tenure', 'emi_eligibility',
       'max_monthly_emi'],
      dtype='object')

In [5]:
for i in numerical_cols:
    print(i,"Unique value:",df[i].nunique())

age Unique value: 15
monthly_salary Unique value: 12780
years_of_employment Unique value: 356
monthly_rent Unique value: 4396
family_size Unique value: 5
dependents Unique value: 5
school_fees Unique value: 132
college_fees Unique value: 202
travel_expenses Unique value: 284
groceries_utilities Unique value: 544
other_monthly_expenses Unique value: 373
current_emi_amount Unique value: 508
credit_score Unique value: 339
bank_balance Unique value: 10614
emergency_fund Unique value: 5486
requested_amount Unique value: 1491
requested_tenure Unique value: 76
max_monthly_emi Unique value: 15383


Create derived financial ratios

1.1 disposable_income [Money left after all costs — most predictive for max EMI]

In [7]:
df['total_expenses'] = (
    df['college_fees'] +
    df['monthly_rent'] +
    df['school_fees'] +
    df['travel_expenses'] +
    df['groceries_utilities'] +
    df['other_monthly_expenses']
)

In [8]:
df['disposable_income'] = df['monthly_salary'] - df['total_expenses']

debt_to_income_ratio   Industry standard metric. DTI > 50% = very risky

In [9]:
df['debt_to_income'] = df['current_emi_amount'] /df['monthly_salary']

expense_ratio   % of income spent. > 80% = almost no EMI capacity

In [10]:
df['expense_ratio'] = df['total_expenses'] / df['monthly_salary']

emi_to_income_ratio [Forward-looking total EMI burden after new loan]

In [16]:
df['emi_to_income_ratio'] = (df['current_emi_amount'] + df['estimated_new_emi'] )/ df['monthly_salary']

estimated_new_emi  [Converts loan request into monthly cost estimate]

In [13]:
df['estimated_new_emi'] = df['requested_amount'] / df['requested_tenure']

savings_rate [Months of salary saved — financial buffer indicator]

In [17]:
df['savings_rate'] = df['bank_balance'] / df['monthly_salary']

emergency_coverage [Months of expenses covered if income stops]

In [18]:
df['emergency_coverage'] = df['emergency_fund'] / (df['total_expenses'] + 1)

Salary_per_dependent   [Income normalised by financial responsibility]

In [ ]:
df['Salary_per_dependent'] = df['monthly_salary'] / (df['dependents'] + 1)

0         27533.333333
1         10750.000000
2         21525.000000
3         13360.000000
4         14325.000000
              ...     
404795     8100.000000
404796     9840.000000
404797    25700.000000
404798    11800.000000
404799    17450.000000
Name: Salary_per_dependent, Length: 404800, dtype: float64

credit_score_band [Captures credit threshold effects non-linearly]

In [24]:
conditions = [
    df['credit_score'] < 550,
    df['credit_score'] < 650,
    df['credit_score'] < 750
]

choices = [3, 2, 1]

df['credit_risk_band'] = np.select(conditions, choices, default=0)

In [26]:
import pandas as pd
df = pd.read_csv(r'C:\Users\SAKTHI\Desktop\myproject\EMI Predict AI\data\feature_dataset.csv')

df.columns

Index(['Unnamed: 0', 'age', 'gender', 'marital_status', 'education',
       'monthly_salary', 'employment_type', 'years_of_employment',
       'company_type', 'house_type', 'monthly_rent', 'family_size',
       'dependents', 'school_fees', 'college_fees', 'travel_expenses',
       'groceries_utilities', 'other_monthly_expenses', 'existing_loans',
       'current_emi_amount', 'credit_score', 'bank_balance', 'emergency_fund',
       'emi_scenario', 'requested_amount', 'requested_tenure',
       'emi_eligibility', 'max_monthly_emi'],
      dtype='object')

In [ ]:
# Data handling
import pandas as pd
import numpy as np
import os
import joblib
# Preprocessing tools from scikit-learn
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

class FeatureEngineer:
    """
    Production-grade Feature Engineering Pipeline
    for EMI Eligibility & Risk Prediction
    """

    def __init__(self, df: pd.DataFrame):
        print("Feature Engineer Initialized")
        self.df = df.copy()
        self.pipeline = None
        self.feature_names_ = None

    #2. FFEATURE CREATION:

    def create_financial_ratios(self):

        print(" Creation of Feature...")

        self.df['total_expenses'] = (
            self.df['college_fees'] +
            self.df['monthly_rent'] +
            self.df['school_fees'] +
            self.df['travel_expenses'] +
            self.df['groceries_utilities'] +
            self.df['other_monthly_expenses']
        )

        self.df['disposable_income'] = self.df['monthly_salary'] - self.df['total_expenses']

        self.df['debt_to_income'] = self.df['current_emi_amount'] / self.df['monthly_salary']

        self.df['expense_ratio'] = self.df['total_expenses'] / self.df['monthly_salary']

        self.df['estimated_new_emi'] = self.df['requested_amount'] / self.df['requested_tenure']

        self.df['emi_to_income_ratio'] = (self.df['current_emi_amount'] + self.df['estimated_new_emi']) / self.df['monthly_salary']

        self.df['savings_rate'] = self.df['bank_balance'] / self.df['monthly_salary']

        self.df['emergency_coverage'] = self.df['emergency_fund'] / (self.df['total_expenses'] + 1)

        self.df['Salary_per_dependent'] = self.df['monthly_salary'] / (self.df['dependents'] + 1)


        self.conditions = [self.df['credit_score'] < 550,self.df['credit_score'] < 650,self.df['credit_score'] < 750]
        self.choices = [3, 2, 1]
        self.df['credit_risk_band'] = np.select(self.conditions, self.choices, default=0)

        return self
    
    # Encode Categories

    def encode_categories(self, target=None):

        print(" Encoding categorical and scaling numeric features...")

        if target is None:
            target = []

        # Remove target

        ec = self.df.drop(columns=target,errors='ignore')

        # Detect feature types

        categories_feature = ec.select_dtypes(include=['object']).columns.tolist()
        Numerical_feature = ec.select_dtypes(include=["int64", "float64"]).columns.tolist()
        print("Categorical Columns:", categories_feature)
        print("Categorical Columns:", Numerical_feature)

        # 

        preprocessor = ColumnTransformer(transformers=[
            ("categories_feature",OneHotEncoder(handle_unknown='ignore'),categories_feature),
            ("Numerical_feature",StandardScaler(),Numerical_feature)
        ],
        remainder='drop')

        # 
        
        self.pipeline = Pipeline(steps=[("preprocessor", preprocessor)])
        #

        transformed = self.pipeline.fit_transform(ec)

    def run_full_pipeline(self, target_cols=None):

        print("\n🚀 Running full feature engineering pipeline...\n")
        (
            self.create_financial_ratios()
        )

        return self.prepare_for_modeling(target_cols)

if __name__ == "__main__":

    input_path = r"C:\Users\SAKTHI\Desktop\myproject\EMI Predict AI\data\emi_prediction_dataset_eda.csv"
    output_dir = r"C:\Users\SAKTHI\Desktop\myproject\EMI Predict AI\model"
    os.makedirs(output_dir, exist_ok=True)

    df = pd.read_csv(input_path)

    TARGET_COLS = ["emi_eligibility", "max_monthly_emi"]

    fe = FeatureEngineer(df)
    ec = fe.run_full_pipeline(target_cols=TARGET_COLS)

    joblib.dump(fe.pipeline, os.path.join(output_dir, "feature_pipeline.pkl"))
    joblib.dump(fe.feature_names_, os.path.join(output_dir, "feature_columns.pkl"))

    print("✅ Feature pipeline & feature columns saved successfully")
    print("🔢 Total features:", len(fe.feature_names_))